# RAG Evaluation: Answer Metrics, BERTScore and the RAG Triad

A RAG system can fail in retrieval, in generation, or in both — so evaluate in layers, cheapest first:

| Layer | Metrics | Needs |
|---|---|---|
| Retrieval | **Hit rate** — does the context contain the answer? | reference answer |
| Answer overlap | **Exact match, token F1, RapidFuzz** | reference answer |
| Semantic similarity | **BERTScore** | reference answer |
| LLM-as-judge (**RAG Triad**) | **Faithfulness**, **Answer Relevancy**, **Contextual Recall** via DeepEval | judge LLM |

The RAG Triad separates the failure modes: is the *context* relevant, is the *answer grounded* in it, and does the answer address the *question*?

Data comes from the HotpotQA collection built in `6.5-Storage-Vector-Databases.ipynb`. For safety layers (PII, jailbreaks, human approval) see `week8/8.12-Guardrails-Middleware.ipynb`.

In [ ]:
%pip install -q langchain langchain-qdrant langchain-huggingface langchain-openai qdrant-client rapidfuzz tqdm evaluate bert_score deepeval

## 1. Setup

Models, URLs and the Qdrant collection come from `config.yaml` via `rag_common.py`.

In [ ]:
import os
import random
from concurrent.futures import ThreadPoolExecutor
from statistics import mean

import rag_common as rc

cfg = rc.load_config()
rc.check_services(cfg)
qa_pairs, _ = rc.load_hotpotqa(cfg)
client, embeddings, doc_store = rc.connect_qdrant(cfg)
llms = rc.load_llms(cfg)

print(f"{len(qa_pairs)} QA pairs | {client.get_collection(cfg['hotpotqa']['collection']).points_count} vectors | models: {list(llms)}")

## 2. Run the RAG Pipeline

Retrieve contexts **once** and reuse them for every model and metric, so differences come from the generator, not the retriever.

In [ ]:
random.seed(76)
examples = random.sample(qa_pairs, k=min(10, len(qa_pairs)))

retriever = doc_store.as_retriever(search_kwargs={"k": 3})
for ex in examples:
    ex["docs"] = retriever.invoke(ex["question"])

# Models are independent, so evaluate them concurrently (Ollama serves parallel requests)
with ThreadPoolExecutor(max_workers=len(llms)) as pool:
    futures = {name: pool.submit(rc.evaluate_answers, name, llm, examples) for name, llm in llms.items()}
    details = {name: future.result() for name, future in futures.items()}

## 3. Answer-Quality and Retrieval Metrics

**Hit** measures retrieval, the rest measure the generated answer. HotpotQA answers are short strings, so exact match is strict — token F1 and RapidFuzz are more forgiving of phrasing.

In [ ]:
header = f"{'model':<28}{'EM':>7}{'F1':>7}{'Fuzzy':>8}{'Hit':>7}{'Time(s)':>9}"
print(header)
print("-" * len(header))
for name, rows in details.items():
    s = rc.summarize(rows)
    print(f"{name:<28}{s['exact_match']*100:>7.1f}{s['token_f1']*100:>7.1f}{s['rapidfuzz']*100:>8.1f}"
          f"{s['retrieval_hit']*100:>7.1f}{s['inference_time']:>9.2f}")

## 4. BERTScore

[BERTScore](https://github.com/Tiiiger/bert_score) compares prediction and reference with contextual embeddings, so paraphrases score well even with little word overlap. F1 is the headline number. `distilbert-base-uncased` keeps it light; use `roberta-large` for higher fidelity.

In [ ]:
import evaluate as hf_evaluate

bertscore = hf_evaluate.load("bertscore")

header = f"{'model':<28}{'Precision':>11}{'Recall':>9}{'F1':>8}"
print(header)
print("-" * len(header))
for name, rows in details.items():
    scores = bertscore.compute(
        predictions=[r["prediction"] for r in rows],
        references=[r["reference"] for r in rows],
        lang="en", model_type="distilbert-base-uncased", verbose=False,
    )
    print(f"{name:<28}{mean(scores['precision']):>11.3f}{mean(scores['recall']):>9.3f}{mean(scores['f1']):>8.3f}")

## 5. RAG Triad with DeepEval (LLM-as-Judge)

[DeepEval](https://docs.confident-ai.com) scores the whole pipeline with a judge LLM — here a local Ollama model, so no API key is needed.

| Metric | Question it answers | Needs reference? |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? (hallucination check) | No |
| **Answer Relevancy** | Does the answer address the question? | No |
| **Contextual Recall** | Does the retrieved context cover the reference answer? | Yes |

In [ ]:
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

from deepeval import evaluate as deepeval_evaluate
from deepeval.metrics import AnswerRelevancyMetric, ContextualRecallMetric, FaithfulnessMetric
from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase

judge = OllamaModel(model=cfg["ollama"]["model"], base_url=cfg["ollama"]["url"])
metrics = [
    FaithfulnessMetric(model=judge, threshold=0.5, verbose_mode=False),
    AnswerRelevancyMetric(model=judge, threshold=0.5, verbose_mode=False),
    ContextualRecallMetric(model=judge, threshold=0.5, verbose_mode=False),
]

print(f"Judge: {cfg['ollama']['model']}\n")
header = f"{'model':<28}{'Faithfulness':>14}{'Ans. Relevancy':>16}{'Ctx. Recall':>13}"
print(header)
print("-" * len(header))
for name, rows in details.items():
    cases = [
        LLMTestCase(input=r["question"], actual_output=r["prediction"],
                    retrieval_context=r["contexts"], expected_output=r["reference"])
        for r in rows
    ]
    results = deepeval_evaluate(cases, metrics=metrics).test_results
    faith, relev, recall = (mean(t.metrics_data[i].score or 0 for t in results) for i in range(3))
    print(f"{name:<28}{faith:>14.3f}{relev:>16.3f}{recall:>13.3f}")

## 6. Everything We Tried: The Results Ledger

Every retrieval notebook (`6.6` to `6.9`) wrote its results to one shared ledger, scored on the **same HotpotQA question sample**. Here is the whole course in one table. Read it by *stage*: which kind of change moved the metric most, and which did nothing?

In [ ]:
rc.show_ablation()

## Summary

1. **Evaluate in layers** — retrieval hit first, then answer overlap, then semantic and judge-based metrics.
2. **Separate retrieval from generation**: low Contextual Recall points at the retriever; low Faithfulness points at the generator.
3. **Overlap metrics under-reward correct paraphrases**; BERTScore and LLM judges catch them, at higher cost and with judge bias.
4. **Compare techniques on one sample**: the shared ledger shows which change (chunking, hybrid, rerank, query transform) actually helped.
5. **Re-run the suite** whenever the corpus, chunking, retriever or prompt changes, so regressions are visible.

Next: `6.12-RAG-Citations-Abstention-and-Diagnosis.ipynb` adds citations, abstention and stage-wise diagnosis.
